# Model 2.2: Custom Linear Support Vector Machine (Task 3)

## Roadmap

**Sklearn SVM reference → derive hinge subgradient → implement the training loop → vary optimisation parameters → five-fold CV → Kaggle submission**

The final model updates its own weight vector `w` and bias `b`. Sklearn is used only for a reference benchmark, fold construction and macro-F1 calculation.

## 1. Loading the Supplied Dataset

The competition files are loaded directly with `kagglehub` and thee supplied 5,000 TF-IDF columns are then converted to SciPy CSR matrices so the custom SVM can perform memory-efficient sparse matrix operations without changing the supplied features.

In [7]:
# Import packages
from dataclasses import dataclass
from pathlib import Path
import kagglehub
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.svm import LinearSVC

# Define the competition and output path
COMPETITION_NAME = "50-007-machine-learning-may-2026"
competition_path = Path(kagglehub.competition_download(COMPETITION_NAME))

RANDOM_STATE = 42

# Load the supplied competition files
train_features_df = pd.read_csv(competition_path / "train_features.csv")
test_features_df = pd.read_csv(competition_path / "test_features.csv")
sample_submission_df = pd.read_csv(competition_path / "sample_submission.csv")

# Separate the labels, IDs and 5,000 supplied TF-IDF features
y = train_features_df.iloc[:, 1].to_numpy(dtype=np.int8)
test_ids = test_features_df.iloc[:, 0].to_numpy()

# Convert the supplied feature columns to sparse matrices for efficient training
X = csr_matrix(
    train_features_df.iloc[:, 2:].to_numpy(dtype=np.float32, copy=False)
)
X_test = csr_matrix(
    test_features_df.iloc[:, 1:].to_numpy(dtype=np.float32, copy=False)
)

# Display basic dataset information
print(f"Training shape: {X.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Positive-class rate: {y.mean():.4f}")
print(f"Submission columns: {sample_submission_df.columns.tolist()}")

c:\Users\raean\actual-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training shape: (20000, 5000)
Test shape: (6999, 5000)
Positive-class rate: 0.6252
Submission columns: ['id', 'label']


## Experiment 1.1: Sklearn Linear SVM Reference Benchmark

This was used as a benchmark to evaluate the custom model after. Identical five-fold splits and the macro-F1 metric make it possible to check whether the custom implementation
behaves competitively.

In [8]:
# Find the decision threshold that produces the highest macro-F1
def threshold_search(y_true, scores):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.20, 0.60, 241)))
    values = np.asarray([
        f1_score(y_true, scores >= threshold, average="macro")
        for threshold in thresholds
    ])
    index = int(values.argmax())
    return float(thresholds[index]), float(values[index])


# Create five reproducible stratified folds
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
reference_oof = np.zeros(len(y), dtype=np.float64)

# Train the sklearn reference model and collect out-of-fold scores
for training_rows, validation_rows in folds.split(X, y):
    reference_model = LinearSVC(
        C=1.0, loss="hinge", class_weight={0: 1.0, 1: 0.85},
        max_iter=10000, random_state=RANDOM_STATE
    )
    reference_model.fit(X[training_rows], y[training_rows])
    reference_oof[validation_rows] = reference_model.decision_function(
        X[validation_rows]
    )

# Tune and report the reference-model decision threshold
reference_threshold, reference_f1 = threshold_search(y, reference_oof)
print(f"Reference tuned five-fold macro-F1: {reference_f1:.5f}")
print(f"Reference threshold: {reference_threshold:.5f}")

Reference tuned five-fold macro-F1: 0.73985
Reference threshold: 0.19522


## Experiment 2.1: Implementing the Hinge-Loss Subgradient

For signed labels `y` in {-1, +1}, the margin loss for one observation is:

`max(0, 1 - y * (w.T @ x + b))`

If the margin is at least one, its loss subgradient is zero. If the margin is
below one, a valid subgradient with respect to `w` is `-y*x`, and with respect
to `b` it is `-y`. L2 regularisation adds `lambda*w` to the weight
subgradient. The loop below computes these quantities and updates `w` and `b`
directly using a step size proportional to `1/sqrt(epoch)`.

In [9]:
# Store diagnostic information from each model fit
@dataclass
class FitSummary:
    epochs_run: int
    initial_objective: float
    best_objective: float
    final_gradient_norm: float
    finite: bool


class CustomLinearSVM:
    # Fit a Linear SVM using a custom batch subgradient-descent loop
    def __init__(self, regularization=1e-4, learning_rate=120.0,
                 bias_step_scale=0.05, positive_cost=1.0, max_epochs=240,
                 tolerance=1e-7, patience=60):
        self.regularization = float(regularization)
        self.learning_rate = float(learning_rate)
        self.bias_step_scale = float(bias_step_scale)
        self.positive_cost = float(positive_cost)
        self.max_epochs = int(max_epochs)
        self.tolerance = float(tolerance)
        self.patience = int(patience)
        self.coef_ = None
        self.intercept_ = 0.0
        self.objective_history_ = []
        self.fit_summary_ = None

    def _objective_and_subgradient(self, matrix, signed_target, sample_weight,
                                   weights, intercept):
        # Calculate decision scores and identify samples inside the margin
        decision = np.asarray(matrix @ weights).ravel() + intercept
        violation = 1.0 - signed_target * decision
        active = violation > 0.0
        n_samples = len(signed_target)

        # Calculate the weighted hinge loss
        weighted_y = sample_weight[active] * signed_target[active]
        hinge_value = float(
            np.sum(sample_weight[active] * violation[active]) / n_samples
        )

        # Add L2 regularization to obtain the complete objective
        objective = (
            0.5 * self.regularization * float(weights @ weights) + hinge_value
        )

        # Calculate subgradients with respect to the weights and bias
        gradient_w = (
            self.regularization * weights
            - np.asarray(matrix[active].T @ weighted_y).ravel() / n_samples
        )
        gradient_b = -float(np.sum(weighted_y) / n_samples)
        return objective, gradient_w, gradient_b

    def fit(self, matrix, target):
        # Convert labels from 0 and 1 to -1 and +1
        signed_target = np.where(target == 1, 1.0, -1.0)

        # Assign the selected cost to machine-generated samples
        sample_weight = np.where(target == 1, self.positive_cost, 1.0)

        # Initialize the weight vector and bias to zero
        weights = np.zeros(matrix.shape[1], dtype=np.float64)
        intercept = 0.0

        # Track the parameters with the lowest observed objective
        best_weights, best_intercept = weights.copy(), intercept
        best_objective, no_improvement = np.inf, 0
        final_gradient_norm = np.inf

        # Perform full-batch subgradient updates
        for epoch in range(1, self.max_epochs + 1):
            objective, gradient_w, gradient_b = self._objective_and_subgradient(
                matrix, signed_target, sample_weight, weights, intercept
            )
            self.objective_history_.append(objective)

            if objective < best_objective - self.tolerance:
                best_objective = objective
                best_weights, best_intercept = weights.copy(), intercept
                no_improvement = 0
            else:
                no_improvement += 1

            # Reduce the learning rate as training progresses
            step_size = self.learning_rate / np.sqrt(epoch)
            weights -= step_size * gradient_w

            # Apply a separately tuned step scale to the dense bias coordinate
            intercept -= step_size * self.bias_step_scale * gradient_b
            final_gradient_norm = float(np.sqrt(
                float(gradient_w @ gradient_w) + gradient_b ** 2
            ))

            if no_improvement >= self.patience:
                break

        # Restore the parameters associated with the best objective
        self.coef_, self.intercept_ = best_weights, float(best_intercept)
        self.fit_summary_ = FitSummary(
            len(self.objective_history_), float(self.objective_history_[0]),
            float(best_objective), final_gradient_norm,
            bool(np.all(np.isfinite(best_weights)) and np.isfinite(best_intercept))
        )
        return self

    def decision_function(self, matrix):
        # Return the signed distance from the linear decision boundary
        if self.coef_ is None:
            raise RuntimeError("Fit the model before requesting predictions")
        return np.asarray(matrix @ self.coef_).ravel() + self.intercept_

    def predict(self, matrix, threshold=0.0):
        # Convert decision scores into binary labels
        return (self.decision_function(matrix) >= threshold).astype(np.int8)

## Experiment 2.2: Varying Subgradient-Descent Parameters

The screen varies regularisation, the weight learning rate, the bias-step scale
and positive-class cost. Each configuration receives 240 epochs on the same
stratified holdout split. The two strongest configurations advance to CV.

In [10]:
# Create all 16 combinations of the four hyperparameters
configurations = [
    {"regularization": regularization, "learning_rate": learning_rate,
     "bias_step_scale": bias_step_scale, "positive_cost": positive_cost}
    for regularization in (3e-5, 1e-4)
    for learning_rate in (120.0, 240.0)
    for bias_step_scale in (0.02, 0.05)
    for positive_cost in (0.75, 0.85)
]

# Split 80% of the training data for fitting and 20% for validation
training_rows, validation_rows = train_test_split(
    np.arange(len(y)), test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
holdout_results = []

# Train and evaluate every configuration on the same holdout split
for parameters in configurations:
    model = CustomLinearSVM(**parameters, max_epochs=240, patience=60).fit(
        X[training_rows], y[training_rows]
    )
    validation_scores = model.decision_function(X[validation_rows])
    threshold, score = threshold_search(y[validation_rows], validation_scores)
    holdout_results.append({
        **parameters, "holdout_macro_f1": score, "threshold": threshold,
        "epochs": model.fit_summary_.epochs_run,
        "best_objective": model.fit_summary_.best_objective,
    })

# Arrange the screening results from highest to lowest macro-F1
holdout_table = pd.DataFrame(holdout_results).sort_values(
    "holdout_macro_f1", ascending=False
)
holdout_table

,regularization,learning_rate,bias_step_scale,positive_cost,holdout_macro_f1,threshold,epochs,best_objective
4,0.00003,240.0,0.02,0.75,0.745620,0.167085,240,0.456657
6,0.00003,240.0,0.05,0.75,0.744658,0.079301,240,0.458964
7,0.00003,240.0,0.05,0.85,0.744406,0.195565,240,0.479959
5,0.00003,240.0,0.02,0.85,0.744008,0.194641,240,0.478612
13,0.00010,240.0,0.02,0.85,0.741884,0.256236,240,0.539510
12,0.00010,240.0,0.02,0.75,0.741063,0.133083,240,0.516785
15,0.00010,240.0,0.05,0.85,0.740911,0.277588,240,0.540092
14,0.00010,240.0,0.05,0.75,0.740535,0.128857,240,0.517763
1,0.00003,120.0,0.02,0.85,0.734569,0.398753,240,0.535298
9,0.00010,120.0,0.02,0.85,0.734050,0.439204,240,0.570903


## Experiment 3.1: Five-Fold Cross-Validation of the Finalists

Both finalists use the same five folds and receive up to 800 training epochs.
The final decision threshold is learned from pooled out-of-fold predictions.
Training F1 is recorded only as an overfitting diagnostic; selection uses
out-of-fold macro-F1.

In [11]:
# Select the two configurations with the highest holdout macro-F1
finalists = sorted(
    holdout_results, key=lambda row: row["holdout_macro_f1"], reverse=True
)[:2]
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results, test_scores_by_finalist = [], []

# Evaluate each finalist using the same five stratified folds
for parameters in finalists:
    configuration = {key: parameters[key] for key in
        ("regularization", "learning_rate", "bias_step_scale", "positive_cost")}
    oof_scores = np.zeros(len(y), dtype=np.float64)
    test_scores = np.zeros(len(test_ids), dtype=np.float64)
    train_f1_values, validation_f1_values = [], []

    # Train one custom model for each fold
    for training_rows, validation_rows in folds.split(X, y):
        model = CustomLinearSVM(
            **configuration, max_epochs=800, patience=200
        ).fit(X[training_rows], y[training_rows])
        training_scores = model.decision_function(X[training_rows])
        validation_scores = model.decision_function(X[validation_rows])

        # Store validation scores and average test scores across the five models
        oof_scores[validation_rows] = validation_scores
        test_scores += model.decision_function(X_test) / 5

        # Record macro-F1 at the default decision threshold of zero
        train_f1_values.append(f1_score(
            y[training_rows], training_scores >= 0, average="macro"
        ))
        validation_f1_values.append(f1_score(
            y[validation_rows], validation_scores >= 0, average="macro"
        ))

    # Tune one decision threshold using the pooled out-of-fold scores
    threshold, tuned_f1 = threshold_search(y, oof_scores)

    # Store the cross-validation results for this finalist
    cv_results.append({
        **configuration,
        "mean_train_f1": np.mean(train_f1_values),
        "mean_validation_f1": np.mean(validation_f1_values),
        "validation_std": np.std(validation_f1_values),
        "tuned_oof_macro_f1": tuned_f1,
        "threshold": threshold,
    })
    test_scores_by_finalist.append(test_scores)

# Arrange the finalists from highest to lowest tuned OOF macro-F1
cv_table = pd.DataFrame(cv_results).sort_values(
    "tuned_oof_macro_f1", ascending=False
)
cv_table

,regularization,learning_rate,bias_step_scale,positive_cost,mean_train_f1,mean_validation_f1,validation_std,tuned_oof_macro_f1,threshold
1,0.00003,240.0,0.05,0.75,0.814267,0.739096,0.004096,0.740386,0.055864
0,0.00003,240.0,0.02,0.75,0.814955,0.739202,0.004186,0.740029,0.037917


## Five-Fold Results

| Regularisation | Learning rate | Bias scale | Positive cost | Mean train F1 | Mean validation F1 | Tuned OOF macro-F1 |
|---:|---:|---:|---:|---:|---:|---:|
| 3e-05 | 240 | 0.02 | 0.75 | 0.81496 | 0.73920 | 0.74003 |
| 3e-05 | 240 | 0.05 | 0.75 | 0.81427 | 0.73910 | **0.74039** |

The selected custom SVM uses regularisation **3e-05**,
learning rate **240**, bias-step scale
**0.05**, positive cost **0.75**
and threshold **0.05586**. Its tuned five-fold macro-F1 is
**0.74039**.

## Experiment 4.1: Creating the Kaggle Submission

The five test decision scores are averaged, then the out-of-fold threshold is applied.

In [15]:
# Select the finalist with the highest tuned OOF macro-F1
best_index = int(np.argmax([
    result["tuned_oof_macro_f1"] for result in cv_results
]))
best_result = cv_results[best_index]

# Apply its tuned threshold to the averaged test decision scores
test_prediction = (
    test_scores_by_finalist[best_index] >= best_result["threshold"]
).astype(np.int8)

# Create the Kaggle submission using the supplied test IDs
submission_path = "outputs/submission_custom_subgradient_svm.csv"
submission = pd.DataFrame({
    "id": test_features_df.iloc[:, 0].to_numpy(),
    "label": test_prediction,
})

submission.to_csv(submission_path, index=False)

# Display submission information and the first five predictions
print(f"Saved: {submission_path}")
print(f"Rows: {len(submission):,}")
print(f"Predicted positive rate: {submission['label'].mean():.4f}")
submission.head()

Saved: outputs/submission_custom_subgradient_svm.csv
Rows: 6,999
Predicted positive rate: 0.6945


,id,label
0,59218,1
1,37110,1
2,23200,0
3,e3357348-166e-4847-a06d-158b7cd83aa5,1
4,61615,0
